# Top tagging: fine-tune a pretrained small model

This uses the **two-class** top-tagging dataset: 0 = background, 1 = signal.
Train on `train`, select the best checkpoint with `val`, and report held-out `test` metrics.
Install this checkout with `pip install -e ".[examples]"` and select that Python environment as the notebook kernel.
A GPU is recommended. The smoke setting only checks the workflow and does not produce benchmark performance.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from omnilearned.metrics import binary_metrics

# Run from the checkout root or examples/. Install: pip install -e ".[examples]"
ROOT = Path.cwd().resolve()
if ROOT.name == "examples":
    ROOT = ROOT.parent
DATA = Path(os.environ.get("OMNILEARNED_EXAMPLE_DATA", ROOT / "datasets")).resolve()
OUTPUT = Path(os.environ.get("OMNILEARNED_EXAMPLE_OUTPUT", ROOT / "example_outputs")).resolve()
CHECKPOINTS = OUTPUT / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
SMOKE = os.environ.get("OMNILEARNED_EXAMPLE_SMOKE", "0") == "1"

def run(*args):
    subprocess.run([sys.executable, "-m", "omnilearned.cli", *map(str, args)], check=True, cwd=ROOT)

In [ ]:
TAG = "top_small_finetuned"
# A fresh directory avoids mixing evaluation files from different world sizes.
import tempfile
RESULTS = Path(tempfile.mkdtemp(prefix=TAG + "_", dir=OUTPUT))
EPOCHS = 1 if SMOKE else 10
BATCH = 2 if SMOKE else 64

## Data
Set `DATA` to the parent containing `top/train`, `top/val`, and `top/test`.
If files are missing, this downloads the public data. Existing files are reused.
Training below disables further remote checks.

In [ ]:
run("dataloader", "--dataset", "top", "--folder", DATA, "--no-check-remote")

## Pretrained checkpoint
Set `OMNILEARNED_PRETRAINED` (or edit `PRETRAINED`) to a compatible **small** pretrained checkpoint.
If absent, the documented public `pretrain_s` checkpoint is fetched. The model loads compatible
backbone/head weights and initializes the task output layer for two classes; the log reports coverage.
The public legacy checkpoint can have a different set of local-interaction layers from your v2 checkpoint.
For a controlled comparison, use your own v2 small pretraining checkpoint with matching architecture.

In [ ]:
from omnilearned.checkpoint import fetch_pretrained
PRETRAINED = Path(os.environ.get("OMNILEARNED_PRETRAINED", CHECKPOINTS / "best_model_pretrain_s.pt"))
if "OMNILEARNED_PRETRAINED" in os.environ and not PRETRAINED.is_file():
    raise FileNotFoundError(PRETRAINED)
fetch_pretrained(PRETRAINED)

## Training

In [ ]:
args = ["train", "--dataset", "top", "--path", DATA,
        "--output_dir", CHECKPOINTS, "--save-tag", TAG,
        "--size", "small", "--mode", "classifier", "--num-classes", 2,
        "--interaction", "--local-interaction", "--epoch", EPOCHS,
        "--batch", BATCH, "--lr", 1e-05, "--wd", 0.0,
        "--num-workers", 0, "--no-check-remote", "--no-ema", "--seed", 42]
args += ["--fine-tune", "--pretrain-path", PRETRAINED, "--lr-factor", 5.0]
if SMOKE:
    args += ["--iterations", 2, "--val-iterations", 1]
run(*args)

## Evaluation of the best validation checkpoint

In [ ]:
run("evaluate", "--dataset", "top", "--path", DATA,
    "--input_dir", CHECKPOINTS, "--output_dir", RESULTS, "--save-tag", TAG,
    "--mode", "classifier", "--num-classes", 2, "--batch", BATCH,
    "--num-workers", 0, "--no-check-remote")

## Accuracy, AUC and background rejection
The ROC working point uses the first threshold achieving **at least** 70% or 30% signal efficiency.
The actual efficiency is printed because finite samples and ties need not attain the target exactly.
Infinite rejection means zero background events pass in this test sample; it is not evidence of perfect population rejection.

In [ ]:
files = sorted(RESULTS.glob(f"outputs_{TAG}_top_*.npz"))
assert files, "Run evaluation first"
# Merge all rank shards before computing metrics, never average per-rank AUCs.
arrays = []
for file in files:
    with np.load(file) as saved:
        arrays.append((saved["pid"].reshape(-1), saved["prediction"]))
labels = np.concatenate([a[0] for a in arrays])
prediction = np.concatenate([a[1] for a in arrays])
result = binary_metrics(labels, prediction, efficiencies=(0.70, 0.30))
print(f"Accuracy: {result['accuracy']:.6f} | AUC: {result['auc']:.6f}")
for point in result["working_points"]:
    print(f"Target effS={point['target_signal_efficiency']:.0%}: "
          f"effS={point['signal_efficiency']:.6f}, "
          f"effB={point['background_efficiency']:.6g}, "
          f"1/effB={point['background_rejection']:.6g}")
fpr, tpr, _ = roc_curve(labels, prediction[:, 1])
plt.plot(tpr, fpr)
plt.xlabel("Signal efficiency")
plt.ylabel("Background efficiency")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.show()